In [1]:
## 2025.07.21 '-었-'에 대한 논문 작성느라 만듦
from pathlib import Path
dir_path = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치")
v_folder = dir_path / "최신버전" / "v"
sen_folder = dir_path / "최신버전" / "sen"

from datetime import datetime
time = []#걸린 시간 체크 리스트
start_t = datetime.now() #시간 체크
time.append(datetime.now() - start_t) #시간간격 저장
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M") #시작 시간 저장

In [2]:
#1분 21초
import pandas as pd
for file in v_folder.glob('*.csv'):
    print(file)
#%% 1.1 v 파일 읽기: 55초 걸림.
df_v_list = []
for file in v_folder.glob("*.csv"):
    with open(file, "r", encoding = "UTF-8") as f:
        df = pd.read_csv(f, low_memory=False)
        #'Unnamed:'를 포함하는 columns 삭제
        cols_to_drop = [col for col in df.columns if 'Unnamed:' in col]
        df = df.drop(columns=cols_to_drop)
        
        df_v_list.append(df)

df_v_list[2].rename(columns={'docu_id': 'file_id'}, inplace=True)

#%% 1.2 문장 파일 읽어오기
print('%%%%%1.2 문장 파일-sen2 읽기') 
df_sen_list = []
for file in sen_folder.glob('*.csv'):
    print(file)
    
for file in sen_folder.glob("*.csv"):
    with open(file, "r", encoding = "UTF-8") as f:
        df = pd.read_csv(f, low_memory=False)
        #'Unnamed:'를 포함하는 columns 삭제
        cols_to_drop = [col for col in df.columns if 'Unnamed:' in col]
        df = df.drop(columns=cols_to_drop)
        df_sen_list.append(df)
df_sen_list[2].rename(columns={'docu_id': 'file_id'}, inplace=True)

for i, df in enumerate(df_v_list):
    df['copus'] = i
    df.loc[df['V_No'] == 3035, 'V_No'] = 3025 #갖다, 가지다를 하나로 합함.
    print(f"{i}, {len(df[df['V_label']=="VX"])}")

C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\국립국어원 구어 말뭉치(버전 1.2)_일부(1,15)_v5.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\국립국어원 일상대화 말뭉치2020,2021_v5.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\v\세종_문어_형태분석_말뭉치_v5.csv
%%%%%1.2 문장 파일-sen2 읽기
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\국립국어원 구어 말뭉치(버전 1.2)_일부(1,15)_sen2.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\국립국어원 일상대화 말뭉치2020,2021_sen2.csv
C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\최신버전\sen\세종_문어_형태분석_말뭉치_sen2.csv
0, 437773
1, 434088
2, 574023


In [3]:
#누락값 채움. 
#object를 category형으로 바꿈. : 45초 걸림
exclude_cols = ['form/label', 'N_form', 'V_form', 'f_V_form']  # 제외할 열
for df in df_v_list:
    for col in df.columns:
        if col in exclude_cols:
            continue  # 제외할 열은 건너뛰기

        # 숫자형이면 -1로 채움
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(-1)

        # 문자열(object)이면 'NULL'로 채우고 고유값 적을 때만 category로
        elif pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].fillna('NULL')
            unique_ratio = df[col].nunique(dropna=False) / len(df)
            if unique_ratio < 0.5:
                df[col] = df[col].astype('category')

In [4]:
import numpy as np

for i, df in enumerate(df_v_list):
    # EP_form
    df['EP_form'] = df['EP_form'].str.replace(' + ', '', regex=False)

    df['EP_TT'] = df['EP_form'].str.contains(r'(었었|았었)', regex=True, na=False)
    df['EP_TT'] = df['EP_TT'].astype(bool)

    df['EP_T'] = (
        (~df['EP_TT']) &
        df['EP_form'].str.contains(r'(었|았)', regex=True, na=False)
    )

    df['EP_M'] = df['EP_form'].str.contains(r'(겠|겄)', regex=True, na=False)

    # f_EP_form
    df['f_EP_form'] = df['f_EP_form'].str.replace(' + ', '', regex=False)

    df['f_EP_TT'] = df['f_EP_form'].str.contains(r'(었었|았었)', regex=True, na=False)
    df['f_EP_TT'] = df['f_EP_TT'].astype(bool)

    df['f_EP_T'] = (
        (~df['f_EP_TT']) &
        df['f_EP_form'].str.contains(r'(었|았)', regex=True, na=False)
    )

    df['f_EP_M'] = df['f_EP_form'].str.contains(r'(겠|겄)', regex=True, na=False)

    df['self_VX'] = np.where(df['vx_no.'] == -1, False, True)

C:\Users\yu2hy\AppData\Local\Temp\ipykernel_27800\223830516.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['EP_TT'] = df['EP_form'].str.contains(r'(었었|았었)', regex=True, na=False)
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_27800\223830516.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['EP_form'].str.contains(r'(었|았)', regex=True, na=False)
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_27800\223830516.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['EP_M'] = df['EP_form'].str.contains(r'(겠|겄)', regex=True, na=False)
C:\Users\yu2hy\AppData\Local\Temp\ipykernel_27800\223830516.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use st

In [5]:
df_v_list[2].head(100)

,ID,sen_id,word_id,word_id2,form/label,N_form,N_label,V_form,V_label,EP_form,...,V_label_0,V_No,copus,EP_TT,EP_T,EP_M,f_EP_TT,f_EP_T,f_EP_M,self_VX
0,12,AA0001.7,10,10,넓히/VV + 어/EF,NaN,NULL,넓히,VV,NULL,...,VV,3743,2,False,False,False,False,False,False,False
1,14,AA0001.8,2,2,세계/NNG + 적/XSN + 이/VCP + ㄴ/ETM,세계 + 적,XSN,이,VCP,NULL,...,VP,1001,2,False,False,False,False,False,False,False
2,23,AA0001.8,11,11,나서/VV + 었/EP + 다/EF + ./SF,NaN,NULL,나서,VV,었,...,VV,3132,2,False,True,False,False,True,False,False
3,28,AA0001.9,5,5,사용하/VV + 는/ETM,NaN,NULL,사용하,VV,NULL,...,VV,3083,2,False,False,False,False,False,False,False
4,31,AA0001.9,8,8,디자인하/VV + 아/EC,NaN,NULL,디자인하,VV,NULL,...,VV,4999,2,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,388,AA0001.37,9,9,있/VA + 다/EF + ./SF,NaN,NULL,있,VA,NULL,...,VA,2001,2,False,False,False,False,False,False,False
96,399,AA0001.38,11,11,따르/VV + 아/EC,NaN,NULL,따르,VV,NULL,...,VV,3028,2,False,False,False,False,False,False,False
97,401,AA0001.38,13,13,6/SN + 천/NR + 원/NNB + 이/VCP + 다/EF + ./SF,6 + 천 + 원,NNB,이,VCP,NULL,...,VP,1001,2,False,False,False,False,False,False,False
98,405,AA0001.39,4,4,2/SN + 배/NNG + 이/VCP + ㄴ/ETM,2 + 배,NNG,이,VCP,NULL,...,VP,1001,2,False,False,False,False,False,False,False


In [6]:
df_sen_list[1].head(10)

,ID,sen_id,form,original_form,birthplace,principal_residence,current_residence,age,sex,education,occupation,file_id,docu_num,category,공공성,상호작용성,화자수,year,S_Len
0,1,SDRW2000000001.1.1.1,저는 여행 다니는 것을,저는 여행 다니는 것을,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,4
1,2,SDRW2000000001.1.1.2,굉장히 좋아하는데요.,굉장히 좋아하는데요.,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,2
2,3,SDRW2000000001.1.1.3,그래가지고,그래가지고,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,2
3,4,SDRW2000000001.1.1.4,스페인이나 뭐 영국,스페인이나 뭐 영국,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,3
4,5,SDRW2000000001.1.1.5,유럽,유럽,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,1
5,6,SDRW2000000001.1.1.6,아니면 국내에서도 뭐 강릉이나,아니면 국내에서도 뭐 강릉이나,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,4
6,7,SDRW2000000001.1.1.7,전주 같은 데를 많이 다녔는데,전주 같은 데를 많이 다녔는데,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,5
7,8,SDRW2000000001.1.1.8,혹시 여행 다니는 거,혹시 여행 다니는 거,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,4
8,9,SDRW2000000001.1.1.9,좋아하시나요?,좋아하시나요?,경기,경기,경기,20대,여성,대재,학생,SDRW2000000001,1,일상대화,사적,대화,2인,2020,1
9,10,SDRW2000000001.1.1.10,저 여행 다니는 거 되게 좋아해서,저 여행 다니는 거 되게 좋아해서,경기,경기,경기,20대,여성,대졸,전문가 및 관련 종사자,SDRW2000000001,1,일상대화,사적,대화,2인,2020,6


In [7]:
for i, df in enumerate(df_v_list):
    print(f"{i}: {len(df)}, v: {len(df[df['V_No']!= -1])}, E, {len(df[df['EN_No']!= -1])}, v-n: {len(df[df['N_form']==None])}" )

0: 3698172, v: 3669128, E, 3690747, v-n: 0
1: 3607347, v: 3587758, E, 3596121, v-n: 0
2: 4599514, v: 4574704, E, 4597601, v-n: 0


In [5]:
import numpy as np

### 추가 정보 붙이는 함수 선언     return result_df
def process_dataframe(result_df):    
    # 피벗테이블을 일반 데이터프레임으로 변환
    if 'index' in result_df.columns:
        result_df = result_df.reset_index()
    
    # category 덧붙이기
    if ('category' in result_df.columns) & ('category_0' not in result_df.columns):    
        data = {
            "category": ["강의", "강의, 학술집담회", "낭독", "뉴스", "방송대화", "방송인터뷰", "토론",
                        "교육자료", "사회일반", "신문", "인문", "일상대화", "자연", "잡지", "체험기술",
                        "총류,일반", "허구", "협력적대화"],
            "category_0": ["공적구어", "공적구어", "공적구어", "공적구어", "공적구어", "공적구어", "공적구어",
                        "문어", "문어", "문어", "문어", "사적구어", "문어", "문어", "문어",
                        "문어", "문어", "사적구어"]
        }
        df_category = pd.DataFrame(data)
        result_df = pd.merge(result_df, df_category, on="category", how="left")

    # V_label 덧붙이기
    if ('V_label' in result_df.columns) & ('V_label_0' not in result_df.columns):
        data = {
            "V_label": ["VV", "VA", "VCN", "VCP", "XSV", "XSA", "VX"],
            "V_label_0": ["VV", "VA", "VA", "VP", "VV", "VA", "VX"]
        }
        df_v_label = pd.DataFrame(data)
        result_df = pd.merge(result_df, df_v_label, on="V_label", how="left")

    # EN 정보 덧붙이기
    if ('EN_No' in result_df.columns) & ('EN_No_0' not in result_df.columns):
        result_df['EN_No'] = pd.to_numeric(result_df['EN_No'], errors='coerce')  # 'EN_No' 열을 숫자형으로 변환, 변환할 수 없는 값은 NaN으로 처리
        result_df['EN_No_0'] = result_df['EN_No'].apply(lambda x: int(x) if not np.isnan(x) else np.nan)  # 'EN_No_0' 열을 추가 (EN_No 내림)
        result_df['EN_No_1'] = result_df['EN_No'] - result_df['EN_No_0']  # 소수점 이하만 남김

    # f_EN 정보 덧붙이기
    if ('f_EN_No' in result_df.columns) & ('f_EN_No_0' not in result_df.columns):
        result_df['f_EN_No'] = pd.to_numeric(result_df['f_EN_No'], errors='coerce')  # 'EN_No' 열을 숫자형으로 변환, 변환할 수 없는 값은 NaN으로 처리
        result_df['f_EN_No_0'] = result_df['f_EN_No'].apply(lambda x: int(x) if not np.isnan(x) else np.nan)  # 'EN_No_0' 열을 추가 (EN_No 내림)
        result_df['f_EN_No_1'] = result_df['f_EN_No'] - result_df['f_EN_No_0']  # 소수점 이하만 남김
        
    # EP 정보 덧붙이기
    if ('EP_form' in result_df.columns) & ('T' not in result_df.columns):
        result_df['T'] = result_df['EP_form'].str.contains('았|었')
        result_df['M'] = result_df['EP_form'].str.contains('겠|겄')
        result_df['H'] = result_df['EP_form'].str.contains('시')

    # f_EP 정보 덧붙이기
    if ('f_EP_form' in result_df.columns) & ('f_T' not in result_df.columns):
        result_df['f_T'] = result_df['f_EP_form'].str.contains('았|었')
        result_df['f_M'] = result_df['f_EP_form'].str.contains('겠|겄')
        result_df['f_H'] = result_df['f_EP_form'].str.contains('시')

    # V_No 정보 덧붙이기
    if ('V_No' in result_df.columns) & ('V_label' not in result_df.columns):
        # V_label 생성: V_No 값에 따라 부여
        def assign_v_label(v_no):
            if 0 <= v_no < 1000:
                return 'VX'
            elif 1000 <= v_no < 2000:
                return 'VC'
            elif 2000 <= v_no < 3000:
                return 'VA'
            elif 3000 <= v_no < 5000:
                return 'VV'
            else:
                return ''  # 혹시 범위 벗어난 경우 대비

        result_df['V_label'] = result_df['V_No'].apply(assign_v_label)
    
    return result_df

### 여러 DataFrame에서 지정한 index_columns 기준으로 피벗 통계를 계산.###    return result_df
def generate_pivot_statistics(df_list, index_columns, filter_fn=None, process_fn=process_dataframe):
    """
    여러 DataFrame에서 지정한 index_columns 기준으로 피벗 통계를 계산.

    Parameters:
        df_list (list of pd.DataFrame): 분석 대상 데이터프레임 리스트
        index_columns (list of str): 피벗 기준 열 목록
        process_fn (callable, optional): 결과 df 후처리 함수 (예: process_dataframe)
        filter_fn (callable, optional): 각 df에 필터를 적용할 함수 (예: lambda df: df[df['V_label'] == 'VX'])

    Returns:
        pd.DataFrame: 통합된 피벗 통계 결과
    """
    result_df_list = []
    print(f"*** 통계를 만듭니다. ***")
    
    for i, df in enumerate(df_list):
        print(f"{i}. 시작합니다.")

        # 필터 함수가 있다면 적용
        print(f"*** {i}. 필터를 적용합니다.***")
        if filter_fn:
            df = filter_fn(df)
        
        print(f"{i}. 피벗 생성 중...")
        cross_0 = pd.pivot_table(
            df[['ID'] + index_columns],
            index=index_columns,
            aggfunc='count',
            fill_value=0,
            dropna=True,
            observed=False
        )
        cross_0 = cross_0[cross_0['ID'] != 0][['ID']]
        result_df_list.append(cross_0)

    result_df = pd.concat(result_df_list).reset_index()

    if process_fn:
        result_df = process_fn(result_df)

    return result_df

###DataFrame에서 필터를 수행한 결과를 문장과 합해서 출력 ##return result_df_list
def generate_sentence_df(df_list, df_sen_list, filter_fn=None, window=0, process_fn=None):
    """
    DataFrame에서 필터를 수행한 결과를 문장과 합해서 출력

    Parameters:
        df_list (list of pd.DataFrame): 분석 대상 데이터프레임 리스트
        df_sen_list (list of pd.DataFrame): 합할 문장 데이터프레임 리스트
        process_fn (callable, optional): 결과 df 후처리 함수 (예: process_dataframe)
        filter_fn (callable, optional): 각 df에 필터를 적용할 함수 (예: lambda df: df[df['V_label'] == 'VX'])

    Returns:
        result_df_list: list of pd.DataFrame: 통합된 데이터프레임 리스트 결과
    """
    result_df_list = []
    print(f"*** 문장 통합 df를 만듭니다. ***")
    
    for i, (df, df_sen) in enumerate(zip(df_list, df_sen_list)):
        print(f"{i}. 통합 중...")

        # 필터 함수가 있다면 적용
        if filter_fn:
            taget_df = filter_fn(df)

            if window > 0:
                df = expand_window(taget_df, df, window)
            else: 
                df = taget_df

        df_sen = df_sen[['sen_id', 'form']]
        result_df = pd.merge(df_sen, df, how='right', on = 'sen_id')

        if process_fn:
            result_df = process_fn(result_df)
        
        print(f"{i}. 통합 완료..., 결과: {len(result_df)}")

        result_df_list.append(result_df)
    return result_df_list

###오류 후보 행들의 sen_id와 word_id2를 기준으로 주변 window만큼 확장한 행을 추출.    #return result_df
def expand_window(taget_df, df, window=2):
    """
    오류 후보 행들의 sen_id와 word_id2를 기준으로 주변 window만큼 확장한 행을 추출.
    """
    # 오류 후보들만 뽑고
    taget_row = taget_df[['sen_id', 'word_id2']].dropna().copy()
    taget_row['word_id2'] = taget_row['word_id2'].astype(int)

    # window 범위로 확장
    expanded_rows = []
    for _, row in taget_row.iterrows():
        sid = row['sen_id']
        wid = row['word_id2']
        for offset in range(-window, window + 1):
            expanded_rows.append((sid, wid + offset))

    # 중복 제거
    expanded_df = pd.DataFrame(expanded_rows, columns=['sen_id', 'word_id2']).drop_duplicates()

    # 기준 df와 merge
    df['word_id2'] = df['word_id2'].astype('Int64')  # Null 허용 정수로
    result_df = pd.merge(df, expanded_df, on=['sen_id', 'word_id2'], how='inner')
    result_df = result_df.sort_values(by=['sen_id', 'word_id2']).reset_index(drop=True)

    return result_df

In [ ]:
### 선어말어미나 vx부류와 결합하지 않은 절의 동사와 어미 결합 확인.
prefix = "V_EF"
save_folder = '2025.07.21-었'
index_columns = ['V_No', 'f_EP_form','f_EN_No', 'f_EN_label', 'copus']

def filter(df):
    return df[(df['V_No']!=-1) &(df['f_EN_No']!=-1)&(df['vx_len']==0) & (df['vx_no.']==-1)& 
              (df['f_EN_No']>=1000)& (df['f_EN_No']<2000)& (df['f_EN_label'] == "EF")]

result_df = generate_pivot_statistics(
    df_list=df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")


*** 통계를 만듭니다. ***
0. 피벗 생성 중...
1. 피벗 생성 중...
2. 피벗 생성 중...
***저장합니다.:    2025-07-31 20:49:59.169979***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_EF_statistics_2025-07-31_20-49.csv (73,804행) ***


In [ ]:
### 선어말어미나 vx부류와 결합하지 않은 절의 동사와 어미 결합 확인. 문장 출력
result_df_list = generate_sentence_df(
    df_list=df_v_list,
    df_sen_list=df_sen_list,
    process_fn=process_dataframe,
    filter_fn=filter,
    window=0
)
## 문장 파일 각각 저장하기.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")

for i, result_df in enumerate(result_df_list):
    out_file = dir_path / save_folder / f'{prefix}_sentence_{i}_{timestamp}.csv'
    result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
    print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 문장 통합 df를 만듭니다. ***
0. 통합 중...
0. 통합 완료..., 결과: 585974
1. 통합 중...
1. 통합 완료..., 결과: 460953
2. 통합 중...
2. 통합 완료..., 결과: 661172
***저장합니다.:    2025-07-25 22:06:59.427015***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_EF_sentence_0_2025-07-25_22-06.csv (585,974행) ***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_EF_sentence_1_2025-07-25_22-06.csv (460,953행) ***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_EF_sentence_2_2025-07-25_22-06.csv (661,172행) ***


In [ ]:
##문장 파일 통합해서 저장하기
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
result_df = pd.concat(result_df_list)
out_file = dir_path / save_folder / f'{prefix}_sentence_{i}_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')

In [ ]:
### 선어말어미나 vx부류와 '-었-' 결합, EF중에 가장 word_id낮은 것. 
prefix = "VX_EP"
save_folder = '2025.07.21-었'
index_columns = ['V_No', 'f_EP_form','f_EN_No', 'f_EN_label', 'copus']

def filter(df):
    # 기본 조건 필터링
    cond = (
        (df['V_No'] != -1) &
        (df['f_EN_No'] != -1) &
        (df['vx_len'] == 0) &
        (df['vx_no.'] == -1) &
        (df['f_EN_No'] >= 1000) &
        (df['f_EN_No'] < 2000) &
        (df['f_EN_label'] == "EF")
    )
    
    df_filtered = df[cond]
    
    if len(df_filtered) == 0:
        return df_filtered  # 빈 DataFrame 반환
    
    # s_id별로 word_id 최대값인 EF만 선택
    return df_filtered.loc[df_filtered.groupby('sen_id', observed=True)['word_id'].idxmax()]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 피벗 생성 중...
1. 피벗 생성 중...
2. 피벗 생성 중...
***저장합니다.:    2025-07-31 20:28:15.506174***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\VX_EP_statistics_2025-07-31_20-28.csv (68,725행) ***


In [26]:
### 선어말어미나 vx부류와 '-었-' 결합, EF중에 가장 word_id낮은 것. 
prefix = "V_EP_VX"
save_folder = '2025.07.21-었'
index_columns = ['V_No','EP_form','vx1_No','vx_len', 'copus']

def filter(df):
    return df[(df['V_No']!=-1) &(df['EN_No']!=-1)]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 피벗 생성 중...
1. 피벗 생성 중...
2. 피벗 생성 중...
***저장합니다.:    2025-08-03 08:47:05.233931***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_EP_VX_statistics_2025-08-03_08-47.csv (163,340행) ***


In [32]:
### 선어말어미나 vx부류와 '-었-' 결합, EF중에 가장 word_id낮은 것. 
prefix = "VX_EP_EN"
save_folder = '2025.07.21-었'
index_columns = ['vx_no.','EP_form','EN_label', 'EN_No','vx_len', 'copus']

def filter(df):
    return df[(df['V_No']!=-1) &(df['EN_No']!=-1)]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 피벗 생성 중...
1. 피벗 생성 중...
2. 피벗 생성 중...
***저장합니다.:    2025-08-05 15:19:26.803922***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\VX_EP_EN_statistics_2025-08-05_15-19.csv (40,083행) ***


In [ ]:
### vx부류와 '-었-' 결합 앞뒤
prefix = "V_EP_VX_EP"
save_folder = '2025.07.21-었'
index_columns = ['EP_form','vx1_No','f_EP_form','copus']

def filter(df):
    return df[(df['vx_no.'] == -1) & (df['vx_len'] == 1) &(df['V_No']!=-1) &(df['EN_No']!=-1)]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 피벗 생성 중...
1. 피벗 생성 중...
2. 피벗 생성 중...
***저장합니다.:    2025-08-03 09:29:11.811295***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_EP_VX_EP_statistics_2025-08-03_09-29.csv (3,075행) ***


In [35]:
### v에 따른 vx부류 결합
prefix = "V_EP_VX_(1000EF)"
save_folder = '2025.07.21-었'
index_columns = ['V_No', 'vx_no.','vx1_No','vx_len', 'copus']

def filter(df):
    return df[(df['V_No']!=-1) &(df['EN_No']!=-1) & (df['f_EN_No']!=-1) & #동사나 어미가 없는 것 제외
              (df['f_EN_No']>=1000)& (df['f_EN_No']<2000)& (df['f_EN_label'] == "EF")] ##EF만


result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 피벗 생성 중...
1. 피벗 생성 중...
2. 피벗 생성 중...
***저장합니다.:    2025-08-06 13:36:34.604407***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_EP_VX_(1000EF)_statistics_2025-08-06_13-36.csv (93,154행) ***


In [7]:
### v에 따른 vx부류 결합
prefix = "V_VX_EP(세종문어_1101)"
save_folder = '2025.07.21-었'
index_columns = ['file_id', 'self_VX', 'V_No', 'vx1_No','vx_len', 'f_EP_T', 'f_EP_M','f_EP_TT']

def filter(df):
    df_copy = df.copy()

    # --- 3) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        df_copy['f_EN_No'].astype(str).str.startswith('1101', na=False) &
        (df_copy['f_EN_label'] == "EF")
    )

    # --- 1) V_No: low-frequency bucketing per range ---
    def bucket_low_freq(col, lo, hi, cap, min_freq):
        mask = df_copy[col].between(lo, hi)
        if mask.any():
            counts = df_copy.loc[mask, col].value_counts()
            low_vals = counts[counts <= min_freq].index
            df_copy.loc[mask & df_copy[col].isin(low_vals), col] = cap

    # VA (2001~2999) -> 2999
    bucket_low_freq('V_No', 2001, 2999, 2999, 1000)
    # VV (3001~4999) -> 4999
    bucket_low_freq('V_No', 3001, 4999, 4999, 1000)

    # --- 2) vx1_No: low-frequency bucketing per range ---
    bucket_low_freq('vx1_No', 101, 199, 199, 1000)   # 상
    bucket_low_freq('vx1_No', 201, 499, 499, 1000)   # 부정/수혜/피사동 등
    bucket_low_freq('vx1_No', 500, 999, 999, 1000)   # 양태


    return df_copy[cond]

result_df = generate_pivot_statistics(
    df_list= [df_v_list[2]],
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...


MemoryError: Unable to allocate 9.26 GiB for an array with shape (1243099008,) and data type uint64

In [9]:
len(result_df)

294472

In [6]:
### 문장 출력

def filter(df):
    df_copy = df.copy()

    # --- 0) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        df_copy['f_EN_No'].astype(str).str.startswith('1101', na=False) &
        (df_copy['f_EN_label'] == "EF")
    )

    # --- 1) V_No: low-frequency bucketing per range ---
    def bucket_low_freq(col, lo, hi, cap, min_freq):
        mask = df_copy[col].between(lo, hi)
        if mask.any():
            counts = df_copy.loc[mask, col].value_counts()
            low_vals = counts[counts <= min_freq].index
            df_copy.loc[mask & df_copy[col].isin(low_vals), col] = cap

    # VA (2001~2999) -> 2999
    bucket_low_freq('V_No', 2001, 2999, 2999, 1000)
    # VV (3001~4999) -> 4999
    bucket_low_freq('V_No', 3001, 4999, 4999, 1000)

    # --- 2) vx1_No: low-frequency bucketing per range ---
    bucket_low_freq('vx1_No', 101, 199, 199, 1000)   # 상
    bucket_low_freq('vx1_No', 201, 499, 499, 1000)   # 부정/수혜/피사동 등
    bucket_low_freq('vx1_No', 500, 999, 999, 1000)   # 양태

    cond = (
        (df_copy['V_No'] != 2999) &
        (df_copy['V_No'] != 4999) &
        (df_copy['vx1_No'] != 199) &
        (df_copy['vx1_No'] != 499) &
        (df_copy['vx1_No'] != 999)
        
    )

    return df_copy[cond]


result_df_list = generate_sentence_df(
    df_list=[df_v_list[2]],
    df_sen_list=[df_sen_list[2]],
    process_fn=process_dataframe,
    filter_fn=filter,
    window=0
)
## 문장 파일 각각 저장하기.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")

for i, result_df in enumerate(result_df_list):
    out_file = dir_path / save_folder / f'{prefix}_sentence_{i}_{timestamp}.csv'
    result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
    print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 문장 통합 df를 만듭니다. ***
0. 통합 중...
0. 통합 완료..., 결과: 3497833
***저장합니다.:    2025-08-09 17:52:48.331982***


NameError: name 'save_folder' is not defined

In [6]:
def bucket_low_freq(sub_df, col, lo, hi, cap, min_freq=None, keep_codes=None):
    """
    지정한 범위(lo~hi) 내에서 값들을 빈도 조건에 따라 cap으로 묶음.
    
    Parameters
    ----------
    sub_df : pd.DataFrame
        작업할 데이터프레임 (subset)
    col : str
        처리할 열 이름
    lo, hi : int
        처리할 값 범위 (inclusive)
    cap : int
        묶을 때 치환할 값
    min_freq : int or None, optional
        None이면 범위 안의 모든 값을 cap으로 치환 (keep_codes 제외).
        숫자면 빈도 <= min_freq인 값만 cap으로 치환.
    keep_codes : list or set, optional
        절대 cap으로 묶이지 않고 그대로 남길 코드 목록
    """
    mask = sub_df[col].between(lo, hi)
    if not mask.any():
        return
    
    if min_freq is None:
        # 범위 안의 값 전부 치환 (keep_codes 제외)
        if keep_codes is not None:
            mask &= ~sub_df[col].isin(keep_codes)
        sub_df.loc[mask, col] = cap
    else:
        counts = sub_df.loc[mask, col].value_counts()
        low_vals = counts[counts <= min_freq].index
        if keep_codes is not None:
            low_vals = [x for x in low_vals if x not in keep_codes]
        sub_df.loc[mask & sub_df[col].isin(low_vals), col] = cap


In [16]:
### v에 따른 vx부류 결합
prefix = "V_VX_EP_500_EN"
save_folder = '2025.07.21-었'
index_columns = ['copus', 'self_VX', 'V_No', 'vx1_No','vx_len', 'f_EP_T', 'f_EP_M','f_EP_TT', 'f_EN_No', 'f_EN_label']

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        (df_copy['f_EN_label'] != "NULL")
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[cond, ['V_No', 'vx1_No', 'f_EN_No']].copy()

    V_min_freq = 500
    # VA (2001~2999) -> 2999
    bucket_low_freq(sub, 'V_No', 2001, 2999, 2999, min_freq = V_min_freq)
    # VV (3001~4999) -> 4999
    bucket_low_freq(sub, 'V_No', 3001, 4999, 4999, min_freq = V_min_freq)

    aspect_list = [101, 102, 123, 111, 112, 122, 124, 113, 125, 126, 131, 132, 114]
    # --- 2) vx1_No: low-frequency bucketing per range ---
    bucket_low_freq(sub, 'vx1_No', 101, 999, 999, keep_codes=aspect_list)   # 상
    
    # 3) f_EN_No
    bucket_low_freq(sub, 'f_EN_No', 0, 999, 999, keep_codes=[])   # 연결어미
    bucket_low_freq(sub, 'f_EN_No', 1000, 1999, 1999, min_freq=1000)   #종결어미
    bucket_low_freq(sub, 'f_EN_No', 2000, 2999, 2999, keep_codes=[])   #관형사형어미
    bucket_low_freq(sub, 'f_EN_No', 3000, 3999, 3999, keep_codes=[])   #명사형어미

    # 서브셋 결과만 df_copy에 반영
    df_copy.loc[cond, ['V_No', 'vx1_No', 'f_EN_No']] = sub[['V_No', 'vx1_No', 'f_EN_No']]

    return df_copy[cond]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...
1. 시작합니다.
*** 1. 필터를 적용합니다.***
1. 피벗 생성 중...
2. 시작합니다.
*** 2. 필터를 적용합니다.***
2. 피벗 생성 중...
***저장합니다.:    2025-08-16 16:27:19.483104***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_VX_EP_500_EN_statistics_2025-08-16_16-27.csv (214,659행) ***


In [18]:
### v에 따른 vx부류 결합
prefix = "V_file"
save_folder = '2025.07.21-었'
index_columns = ['copus', 'file_id','self_VX', 'V_No', 'vx_len', 'f_EP_T', 'f_EP_M','f_EP_TT']

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    cond = (
       (df['V_No']!=-1) &(df['EN_No']!=-1) & (df['f_EN_No']!=-1)
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[cond, ['V_No']].copy()

    V_min_freq = 500
    # VA (2001~2999) -> 2999
    bucket_low_freq(sub, 'V_No', 2001, 2999, 2999, min_freq = V_min_freq)
    # VV (3001~4999) -> 4999
    bucket_low_freq(sub, 'V_No', 3001, 4999, 4999, min_freq = V_min_freq)

    # 서브셋 결과만 df_copy에 반영
    df_copy.loc[cond, ['V_No']] = sub[['V_No']]

    return df_copy[cond]


result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...
1. 시작합니다.
*** 1. 필터를 적용합니다.***
1. 피벗 생성 중...
2. 시작합니다.
*** 2. 필터를 적용합니다.***
2. 피벗 생성 중...
***저장합니다.:    2025-08-17 00:09:35.502307***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_file_statistics_2025-08-17_00-09.csv (2,653,284행) ***


In [21]:
result_df

,copus,file_id,self_VX,V_No,vx_len,f_EP_T,f_EP_M,f_EP_TT,ID,V_label
0,0,SARW1800000001,False,1001,0,False,False,False,8,VC
1,0,SARW1800000001,False,2001,0,False,False,False,1,VA
2,0,SARW1800000001,False,2005,0,False,False,False,2,VA
3,0,SARW1800000001,False,2006,0,False,False,False,2,VA
4,0,SARW1800000001,False,2008,0,False,False,False,2,VA
...,...,...,...,...,...,...,...,...,...,...
2653279,2,JO0153,True,3002,0,True,False,False,5,VV
2653280,2,JO0153,True,3002,1,False,False,False,8,VV
2653281,2,JO0153,True,3002,2,False,False,False,1,VV
2653282,2,JO0153,True,3007,0,False,False,False,3,VV


In [22]:
for i in range(3):
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    print(f"***저장합니다.:    {datetime.now()}***")
    out_file = dir_path / save_folder / f'{prefix}_statistics_{i}.csv'
    save_df = result_df[result_df['copus'] == i]
    save_df.to_csv(out_file, index=True, encoding='utf-8-sig')
    print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

***저장합니다.:    2025-08-17 00:17:09.481910***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_file_statistics_0.csv (2,653,284행) ***
***저장합니다.:    2025-08-17 00:17:11.356747***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_file_statistics_1.csv (2,653,284행) ***
***저장합니다.:    2025-08-17 00:17:13.473086***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_file_statistics_2.csv (2,653,284행) ***


In [9]:
### v에 따른 vx부류 결합
prefix = "V_VX_EP_500_EN"
save_folder = '2025.07.21-었'
index_columns = ['copus', 'self_VX', 'V_label', 'vx1_No','vx_len', 'f_EP_T', 'f_EP_M','f_EP_TT', 'file_id']

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        (df_copy['f_EN_label'] != "NULL")
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[cond, ['V_No', 'vx1_No', 'f_EN_No']].copy()

    V_min_freq = 500
    # VA (2001~2999) -> 2999
    bucket_low_freq(sub, 'V_No', 2001, 2999, 2999, min_freq = V_min_freq)
    # VV (3001~4999) -> 4999
    bucket_low_freq(sub, 'V_No', 3001, 4999, 4999, min_freq = V_min_freq)

    aspect_list = [101, 102, 123, 111, 112, 122, 124, 113, 125, 126, 131, 132, 114]
    # --- 2) vx1_No: low-frequency bucketing per range ---
    bucket_low_freq(sub, 'vx1_No', 101, 999, 999, keep_codes=aspect_list)   # 상
    
    # 3) f_EN_No
    bucket_low_freq(sub, 'f_EN_No', 0, 999, 999, keep_codes=[])   # 연결어미
    bucket_low_freq(sub, 'f_EN_No', 1000, 1999, 1999, min_freq=1000)   #종결어미
    bucket_low_freq(sub, 'f_EN_No', 2000, 2999, 2999, keep_codes=[])   #관형사형어미
    bucket_low_freq(sub, 'f_EN_No', 3000, 3999, 3999, keep_codes=[])   #명사형어미

    # 서브셋 결과만 df_copy에 반영
    df_copy.loc[cond, ['V_No', 'vx1_No', 'f_EN_No']] = sub[['V_No', 'vx1_No', 'f_EN_No']]

    return df_copy[cond]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...
1. 시작합니다.
*** 1. 필터를 적용합니다.***
1. 피벗 생성 중...
2. 시작합니다.
*** 2. 필터를 적용합니다.***
2. 피벗 생성 중...
***저장합니다.:    2025-08-17 00:39:11.773407***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\V_VX_EP_500_EN_statistics_2025-08-17_00-39.csv (475,595행) ***


In [7]:
### v에 따른 vx부류 결합
prefix = "EP_EN"
save_folder = '2025.07.21-었'
index_columns = ['copus', 'file_id', 'self_VX', 'vx_len', 'f_EP_T', 'f_EP_M','f_EP_TT', 'f_EN_No', 'f_EN_label']

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        (df_copy['f_EN_label'] != "NULL")
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[cond, ['f_EN_No']].copy()
    
    # 3) f_EN_No
    bucket_low_freq(sub, 'f_EN_No', 0, 999, 999, keep_codes=[])   # 연결어미
    bucket_low_freq(sub, 'f_EN_No', 1000, 1999, 1999, min_freq=1000)   #종결어미
    bucket_low_freq(sub, 'f_EN_No', 2000, 2999, 2999, keep_codes=[])   #관형사형어미
    bucket_low_freq(sub, 'f_EN_No', 3000, 3999, 3999, keep_codes=[])   #명사형어미

    # 서브셋 결과만 df_copy에 반영
    df_copy.loc[cond, ['f_EN_No']] = sub[['f_EN_No']]

    return df_copy[cond]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...
1. 시작합니다.
*** 1. 필터를 적용합니다.***
1. 피벗 생성 중...
2. 시작합니다.
*** 2. 필터를 적용합니다.***
2. 피벗 생성 중...
***저장합니다.:    2025-08-18 06:51:22.088449***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\EP_EN_statistics_2025-08-18_06-51.csv (909,229행) ***


In [7]:
### v에 따른 vx부류 결합
prefix = "EP_EN(500)"
save_folder = '2025.07.21-었'
index_columns = ['copus', 'self_VX', 'vx_len', 'f_EP_T', 'f_EP_M','f_EP_TT', 'f_EN_No', 'f_EN_label']

def filter(df):
    df_copy = df.copy()

    # --- 1) your existing filter ---
    cond = (
        (df_copy['V_No'] != -1) &
        (df_copy['f_EN_label'] != "NULL")
    )
    # 서브셋만 떼서 작업 (가독성/안전성 ↑)
    sub = df_copy.loc[cond, ['f_EN_No']].copy()
    
    # 3) f_EN_No
    bucket_low_freq(sub, 'f_EN_No', 0, 999, 999, min_freq=500)   # 연결어미
    bucket_low_freq(sub, 'f_EN_No', 1000, 1999, 1999, min_freq=500)   #종결어미
    bucket_low_freq(sub, 'f_EN_No', 2000, 2999, 2999, min_freq=500)   #관형사형어미
    bucket_low_freq(sub, 'f_EN_No', 3000, 3999, 3999, min_freq=500)   #명사형어미

    # 서브셋 결과만 df_copy에 반영
    df_copy.loc[cond, ['f_EN_No']] = sub[['f_EN_No']]

    return df_copy[cond]

result_df = generate_pivot_statistics(
    df_list= df_v_list,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=filter
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
print(f"***저장합니다.:    {datetime.now()}***")
out_file = dir_path / save_folder / f'{prefix}_statistics_{timestamp}.csv'
result_df.to_csv(out_file, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {out_file} ({len(result_df):,}행) ***")

*** 통계를 만듭니다. ***
0. 시작합니다.
*** 0. 필터를 적용합니다.***
0. 피벗 생성 중...
1. 시작합니다.
*** 1. 필터를 적용합니다.***
1. 피벗 생성 중...
2. 시작합니다.
*** 2. 필터를 적용합니다.***
2. 피벗 생성 중...
***저장합니다.:    2025-08-18 08:09:47.952547***
*** 저장 완료: C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치\2025.07.21-었\EP_EN(500)_statistics_2025-08-18_08-09.csv (7,021행) ***
